In [1]:
import sys
sys.path.insert(0, "..")   # lets the notebook import from the project root

import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from ml.common import (ROOT, CAP, SEED, load_fd, select_sensors,
                       add_features, feature_names, rmse, nasa_score)

# Store experiment data in a database file inside the project
mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")

EXP = "rul-fd001"
if mlflow.get_experiment_by_name(EXP) is None:
    mlflow.create_experiment(EXP, artifact_location=(ROOT / "mlruns").as_uri())
mlflow.set_experiment(EXP)

train, test, rul = load_fd("FD001")
keep = select_sensors(train)
y, groups = train["RUL"], train["unit"]
y_test_true = np.minimum(rul["RUL"].values, CAP)
print(len(keep), "sensors kept")

2026/11/07 21:45:58 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/07 21:45:59 INFO mlflow.store.db.utils: Updating database tables


14 sensors kept


In [2]:
MODEL_PARAMS = dict(
    n_estimators=400, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1,
)
ART_DIR = ROOT / "ml" / "artifacts"
ART_DIR.mkdir(parents=True, exist_ok=True)


def run_and_log(win, use_slope, run_name):
    feats = feature_names(keep, use_slope)
    tr = add_features(train, keep, win, use_slope)
    te = add_features(test, keep, win, use_slope)
    X = tr[feats]

    with mlflow.start_run(run_name=run_name):
        mlflow.set_tags({"model": "xgboost", "dataset": "FD001", "features": "rolling"})
        mlflow.log_params({
            "window": win, "use_slope": use_slope, "n_features": len(feats),
            "rul_cap": CAP, "n_sensors": len(keep), **MODEL_PARAMS,
        })

        # Grouped cross-validation, one metric per fold
        cv = []
        for fold, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=5).split(X, y, groups), 1):
            m = XGBRegressor(**MODEL_PARAMS).fit(X.iloc[tr_idx], y.iloc[tr_idx])
            score = rmse(y.iloc[va_idx], m.predict(X.iloc[va_idx]))
            cv.append(score)
            mlflow.log_metric("fold_rmse", score, step=fold)

        # Final model, evaluated on each test engine's last observed cycle
        model = XGBRegressor(**MODEL_PARAMS).fit(X, y)
        last = te.groupby("unit").tail(1)
        preds = np.clip(model.predict(last[feats]), 0, CAP)

        metrics = {
            "cv_rmse": float(np.mean(cv)),
            "cv_rmse_std": float(np.std(cv)),
            "test_rmse": rmse(y_test_true, preds),
            "test_nasa": nasa_score(y_test_true, preds),
        }
        mlflow.log_metrics(metrics)

        # Save files as artifacts: the model and the test predictions
        model_path = ART_DIR / f"{run_name}.json"
        model.save_model(model_path)
        mlflow.log_artifact(str(model_path))

        pred_path = ART_DIR / f"{run_name}_test_predictions.csv"
        pd.DataFrame({"unit": last["unit"].values, "true_rul": y_test_true,
                      "pred_rul": preds}).to_csv(pred_path, index=False)
        mlflow.log_artifact(str(pred_path))

    return metrics

In [3]:
runs = [
    (10, False, "xgb_win10_noslope"),   # Day 4 baseline
    (20, True,  "xgb_win20_slope"),
    (30, True,  "xgb_win30_slope"),     # best so far
]

for win, slope, name in runs:
    print(name, run_and_log(win, slope, name))

xgb_win10_noslope {'cv_rmse': 18.953657487850855, 'cv_rmse_std': 0.5929395284293687, 'test_rmse': 18.554963056057687, 'test_nasa': 1357.1338127514534}
xgb_win20_slope {'cv_rmse': 15.523880653211682, 'cv_rmse_std': 0.6009937245599671, 'test_rmse': 15.572411217693608, 'test_nasa': 443.2000855498268}
xgb_win30_slope {'cv_rmse': 13.19529395745531, 'cv_rmse_std': 0.5593816607552016, 'test_rmse': 13.111420054979888, 'test_nasa': 257.08004533652206}


## Findings: MLflow tracking

- Each run now stores its parameters (window, slope, model settings, seed), its metrics (CV RMSE, per-fold RMSE, test RMSE, NASA score), the trained model file, and the test predictions. This is more reliable than copying numbers from notebook output into a table.
- The three logged runs reproduce my earlier ablation results, so moving the shared code into `ml/common.py` did not change the pipeline.
- The comparison view shows that CV RMSE falls as the window grows when the slope feature is on (18.95 → 15.52 → 13.20 across the three runs shown).
- Shared code (data loading, features, metrics) now lives in `ml/common.py`, so the GRU/LSTM experiments in Week 2 will use exactly the same data, labels and metrics, and can be compared fairly against these XGBoost runs.
- Local MLflow files (`mlflow.db`, `mlruns/`, `ml/artifacts/`) are gitignored because they are outputs that the code can recreate.